# zgw-posix S3 Visual Demo

This notebook demonstrates how to use **Jupyter** with the **zgw-posix S3 gateway**.

We'll cover:
- Connecting to the S3 endpoint
- Bucket operations (create, list, delete)
- Uploading and downloading objects
- Storing and retrieving images
- Data analysis with S3-stored CSV files
- Visual dashboards showing storage usage

---

## 1. Setup & Configuration

First, let's install required packages and configure our S3 connection.

In [ ]:
# Install required packages
!pip install -q boto3 pandas matplotlib seaborn pillow

In [ ]:
import os
import io
import json
import boto3
from botocore.config import Config
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from datetime import datetime
import random

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

In [ ]:
# S3 Configuration
# When running in Kubernetes, these come from environment variables
# For local testing with podman, adjust as needed

S3_ENDPOINT = os.environ.get('S3_ENDPOINT_URL', 'http://localhost:9000')
AWS_ACCESS_KEY = os.environ.get('AWS_ACCESS_KEY_ID', 'testuser')
AWS_SECRET_KEY = os.environ.get('AWS_SECRET_ACCESS_KEY', 'testuser')

# Create boto3 S3 client
s3_client = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'  # Required but not used by zgw
)

# Also create a resource for higher-level operations
s3_resource = boto3.resource(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

print(f"Connected to S3 endpoint: {S3_ENDPOINT}")
print(f"Using access key: {AWS_ACCESS_KEY[:4]}...")

---
## 2. Bucket Operations

Let's create some buckets and explore the S3 namespace.

In [ ]:
# Define our demo buckets
DEMO_BUCKETS = ['demo-images', 'demo-data', 'demo-reports']

# Create buckets
for bucket_name in DEMO_BUCKETS:
    try:
        s3_client.create_bucket(Bucket=bucket_name)
        print(f"Created bucket: {bucket_name}")
    except s3_client.exceptions.BucketAlreadyOwnedByYou:
        print(f"Bucket already exists: {bucket_name}")
    except Exception as e:
        print(f"Error creating {bucket_name}: {e}")

In [ ]:
# List all buckets
response = s3_client.list_buckets()
buckets = response['Buckets']

print(f"\nTotal buckets: {len(buckets)}\n")
print(f"{'Bucket Name':<30} {'Created'}")
print("-" * 60)
for bucket in buckets:
    created = bucket['CreationDate'].strftime('%Y-%m-%d %H:%M:%S')
    print(f"{bucket['Name']:<30} {created}")

---
## 3. Working with Images

Let's generate, upload, and retrieve images from S3.

In [ ]:
def generate_sample_image(width=400, height=300, pattern='gradient'):
    """Generate a sample image for demonstration."""
    import numpy as np
    
    if pattern == 'gradient':
        # Create a colorful gradient
        x = np.linspace(0, 1, width)
        y = np.linspace(0, 1, height)
        xv, yv = np.meshgrid(x, y)
        
        r = (np.sin(xv * 3.14159 * 2) * 127 + 128).astype(np.uint8)
        g = (np.sin(yv * 3.14159 * 2 + 2) * 127 + 128).astype(np.uint8)
        b = (np.sin((xv + yv) * 3.14159) * 127 + 128).astype(np.uint8)
        
        rgb = np.dstack((r, g, b))
    else:
        # Random noise pattern
        rgb = np.random.randint(0, 255, (height, width, 3), dtype=np.uint8)
    
    return Image.fromarray(rgb)

# Generate and display sample images
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

images = [
    ('gradient', generate_sample_image(pattern='gradient')),
    ('noise1', generate_sample_image(pattern='noise')),
    ('noise2', generate_sample_image(pattern='noise'))
]

for ax, (name, img) in zip(axes, images):
    ax.imshow(img)
    ax.set_title(f'{name}.png')
    ax.axis('off')

plt.suptitle('Sample Images to Upload to S3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Upload images to S3
print("Uploading images to demo-images bucket...\n")

for name, img in images:
    # Convert image to bytes
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    buffer.seek(0)
    
    # Upload to S3
    key = f'samples/{name}.png'
    s3_client.upload_fileobj(
        buffer, 
        'demo-images', 
        key,
        ExtraArgs={'ContentType': 'image/png'}
    )
    print(f"Uploaded: s3://demo-images/{key}")

print("\nAll images uploaded successfully!")

In [ ]:
# Retrieve and display images from S3
print("Retrieving images from S3...\n")

# List objects in demo-images bucket
response = s3_client.list_objects_v2(Bucket='demo-images', Prefix='samples/')
objects = response.get('Contents', [])

fig, axes = plt.subplots(1, len(objects), figsize=(15, 4))
if len(objects) == 1:
    axes = [axes]

for ax, obj in zip(axes, objects):
    # Download image
    response = s3_client.get_object(Bucket='demo-images', Key=obj['Key'])
    img_data = response['Body'].read()
    img = Image.open(io.BytesIO(img_data))
    
    ax.imshow(img)
    ax.set_title(f"Retrieved: {obj['Key'].split('/')[-1]}\nSize: {obj['Size']} bytes")
    ax.axis('off')

plt.suptitle('Images Retrieved from S3', fontsize=14, fontweight='bold', color='green')
plt.tight_layout()
plt.show()

---
## 4. Data Analysis with S3

Store CSV data in S3 and perform analysis with pandas.

In [ ]:
# Generate sample sales data
import numpy as np

np.random.seed(42)
n_records = 500

regions = ['North', 'South', 'East', 'West']
products = ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y', 'Service Z']

data = {
    'date': pd.date_range('2024-01-01', periods=n_records, freq='D').tolist() * (n_records // 500 + 1),
    'region': [random.choice(regions) for _ in range(n_records)],
    'product': [random.choice(products) for _ in range(n_records)],
    'quantity': np.random.randint(1, 100, n_records),
    'unit_price': np.random.uniform(10, 500, n_records).round(2),
}
data['date'] = data['date'][:n_records]
data['revenue'] = (np.array(data['quantity']) * np.array(data['unit_price'])).round(2)

df = pd.DataFrame(data)
print(f"Generated {len(df)} sales records")
df.head(10)

In [ ]:
# Upload DataFrame to S3 as CSV
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False)

s3_client.put_object(
    Bucket='demo-data',
    Key='sales/sales_data.csv',
    Body=csv_buffer.getvalue(),
    ContentType='text/csv'
)

print("Uploaded sales data to s3://demo-data/sales/sales_data.csv")

In [ ]:
# Read data back from S3 and analyze
response = s3_client.get_object(Bucket='demo-data', Key='sales/sales_data.csv')
df_from_s3 = pd.read_csv(io.BytesIO(response['Body'].read()))

print(f"Retrieved {len(df_from_s3)} records from S3")
print(f"\nDataset Summary:")
print(f"  Date range: {df_from_s3['date'].min()} to {df_from_s3['date'].max()}")
print(f"  Total revenue: ${df_from_s3['revenue'].sum():,.2f}")
print(f"  Avg order value: ${df_from_s3['revenue'].mean():,.2f}")

In [ ]:
# Create visualizations from S3 data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Revenue by Region (Pie Chart)
ax1 = axes[0, 0]
region_revenue = df_from_s3.groupby('region')['revenue'].sum()
colors = sns.color_palette('husl', len(region_revenue))
wedges, texts, autotexts = ax1.pie(
    region_revenue, 
    labels=region_revenue.index, 
    autopct='%1.1f%%',
    colors=colors,
    explode=[0.02]*len(region_revenue)
)
ax1.set_title('Revenue by Region', fontsize=12, fontweight='bold')

# 2. Revenue by Product (Bar Chart)
ax2 = axes[0, 1]
product_revenue = df_from_s3.groupby('product')['revenue'].sum().sort_values(ascending=True)
bars = ax2.barh(product_revenue.index, product_revenue.values, color=sns.color_palette('viridis', len(product_revenue)))
ax2.set_xlabel('Revenue ($)')
ax2.set_title('Revenue by Product', fontsize=12, fontweight='bold')
for bar, val in zip(bars, product_revenue.values):
    ax2.text(val + 1000, bar.get_y() + bar.get_height()/2, f'${val:,.0f}', va='center', fontsize=9)

# 3. Daily Revenue Trend (Line Chart)
ax3 = axes[1, 0]
df_from_s3['date'] = pd.to_datetime(df_from_s3['date'])
daily_revenue = df_from_s3.groupby('date')['revenue'].sum()
ax3.fill_between(daily_revenue.index, daily_revenue.values, alpha=0.3)
ax3.plot(daily_revenue.index, daily_revenue.values, linewidth=2)
ax3.set_xlabel('Date')
ax3.set_ylabel('Daily Revenue ($)')
ax3.set_title('Daily Revenue Trend', fontsize=12, fontweight='bold')
ax3.tick_params(axis='x', rotation=45)

# 4. Quantity Distribution (Histogram)
ax4 = axes[1, 1]
ax4.hist(df_from_s3['quantity'], bins=20, edgecolor='white', alpha=0.7, color='steelblue')
ax4.axvline(df_from_s3['quantity'].mean(), color='red', linestyle='--', label=f"Mean: {df_from_s3['quantity'].mean():.1f}")
ax4.set_xlabel('Quantity')
ax4.set_ylabel('Frequency')
ax4.set_title('Order Quantity Distribution', fontsize=12, fontweight='bold')
ax4.legend()

plt.suptitle('Sales Analytics Dashboard (Data from S3)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Storage Usage Dashboard

Visualize what's stored in our S3 buckets.

In [ ]:
def get_bucket_stats(bucket_name):
    """Get statistics for a bucket."""
    total_size = 0
    total_objects = 0
    
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket_name):
        for obj in page.get('Contents', []):
            total_size += obj['Size']
            total_objects += 1
    
    return {
        'bucket': bucket_name,
        'objects': total_objects,
        'size_bytes': total_size,
        'size_kb': total_size / 1024
    }

# Collect stats for all buckets
bucket_stats = []
for bucket in s3_client.list_buckets()['Buckets']:
    stats = get_bucket_stats(bucket['Name'])
    bucket_stats.append(stats)

stats_df = pd.DataFrame(bucket_stats)
print("Bucket Statistics:\n")
print(stats_df.to_string(index=False))

In [ ]:
# Visualize storage usage
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Filter to buckets with data
stats_with_data = stats_df[stats_df['objects'] > 0]

if len(stats_with_data) > 0:
    # 1. Objects per bucket
    ax1 = axes[0]
    colors = sns.color_palette('Set2', len(stats_with_data))
    bars1 = ax1.bar(stats_with_data['bucket'], stats_with_data['objects'], color=colors)
    ax1.set_ylabel('Number of Objects')
    ax1.set_title('Objects per Bucket', fontweight='bold')
    ax1.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars1, stats_with_data['objects']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, str(val), ha='center')

    # 2. Size per bucket
    ax2 = axes[1]
    bars2 = ax2.bar(stats_with_data['bucket'], stats_with_data['size_kb'], color=colors)
    ax2.set_ylabel('Size (KB)')
    ax2.set_title('Storage Size per Bucket', fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars2, stats_with_data['size_kb']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}', ha='center')

    # 3. Storage distribution pie
    ax3 = axes[2]
    ax3.pie(stats_with_data['size_bytes'], labels=stats_with_data['bucket'], 
            autopct='%1.1f%%', colors=colors, explode=[0.02]*len(stats_with_data))
    ax3.set_title('Storage Distribution', fontweight='bold')
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'No data in buckets yet', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()

plt.suptitle('S3 Storage Usage Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Save Report to S3

Generate a summary chart and save it back to S3.

In [ ]:
# Create a summary report image
fig, ax = plt.subplots(figsize=(10, 6))

# Summary statistics text
total_buckets = len(bucket_stats)
total_objects = stats_df['objects'].sum()
total_size = stats_df['size_bytes'].sum()

summary_text = f"""
zgw-posix S3 Storage Summary
{'='*40}

Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Total Buckets:     {total_buckets}
Total Objects:     {total_objects}
Total Storage:     {total_size/1024:.2f} KB

Endpoint: {S3_ENDPOINT}

{'='*40}
Generated by Jupyter Notebook
"""

ax.text(0.5, 0.5, summary_text, transform=ax.transAxes, fontsize=14,
        verticalalignment='center', horizontalalignment='center',
        fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
ax.set_axis_off()

# Save to buffer and upload
img_buffer = io.BytesIO()
fig.savefig(img_buffer, format='PNG', dpi=150, bbox_inches='tight', facecolor='white')
img_buffer.seek(0)

report_key = f"reports/summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
s3_client.upload_fileobj(
    img_buffer,
    'demo-reports',
    report_key,
    ExtraArgs={'ContentType': 'image/png'}
)

print(f"Report saved to s3://demo-reports/{report_key}")
plt.show()

---
## 7. Cleanup (Optional)

Remove demo buckets and objects when done.

In [ ]:
# Uncomment to clean up demo resources

# def delete_bucket_contents(bucket_name):
#     """Delete all objects in a bucket."""
#     bucket = s3_resource.Bucket(bucket_name)
#     bucket.objects.all().delete()
#     print(f"Deleted all objects in {bucket_name}")

# for bucket_name in DEMO_BUCKETS:
#     try:
#         delete_bucket_contents(bucket_name)
#         s3_client.delete_bucket(Bucket=bucket_name)
#         print(f"Deleted bucket: {bucket_name}")
#     except Exception as e:
#         print(f"Error deleting {bucket_name}: {e}")

print("Cleanup code is commented out. Uncomment to delete demo resources.")

---
## Summary

This notebook demonstrated:

1. **S3 Connection** - Connecting to zgw-posix using boto3
2. **Bucket Management** - Creating, listing, and managing buckets
3. **Image Storage** - Uploading and retrieving images from S3
4. **Data Analysis** - Storing CSV data and creating visualizations
5. **Storage Dashboard** - Monitoring bucket usage
6. **Report Generation** - Saving analysis results back to S3

### Next Steps

- Try uploading your own files
- Create more complex data pipelines
- Explore versioning and lifecycle policies
- Integrate with machine learning workflows